<a href="https://colab.research.google.com/github/kjahan/nlp_tips_and_trick/blob/main/notebooks/smart_paraphrasing_with_constrained_beam_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## We demo how to keep some words in paraphrasing intact

Use cases: copywriting and SEO!

https://towardsdatascience.com/smart-paraphrasing-using-constrained-beam-search-in-nlp-9af6fd046e5c


https://huggingface.co/blog/constrained-beam-search

In [1]:
!pip install -q sentencepiece
!pip install -q transformers==4.18.0

     |████████████████████████████████| 1.3 MB 5.3 MB/s 
     |████████████████████████████████| 4.0 MB 5.3 MB/s 
     |████████████████████████████████| 163 kB 51.5 MB/s 
     |████████████████████████████████| 6.6 MB 33.3 MB/s 
     |████████████████████████████████| 880 kB 54.3 MB/s 


In [2]:
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("ramsrigouthamg/t5-large-paraphraser-diverse-high-quality")
tokenizer = AutoTokenizer.from_pretrained("ramsrigouthamg/t5-large-paraphraser-diverse-high-quality")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ("device ",device)
model = model.to(device)

Downloading:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.75G [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/773k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

device  cpu


In [3]:
# Copy writing example - Use a given word in paraphrasing

context = "Supercharge your data science skills by building real world projects."

text = "paraphrase: "+context + " </s>"

input_ids = tokenizer(text, max_length=128, padding=True, return_tensors="pt").input_ids
input_ids = input_ids.to(device)

# Beam search

outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=5,
    max_length=128,
    early_stopping=True,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nNormal beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2280: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  "`max_length` is ignored when `padding`=`True` and there is no truncation strategy. "



Normal beam search

Original:  Supercharge your data science skills by building real world projects.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills.
paraphrasedoutput: By implementing real world projects, you can boost your data science skills.
paraphrasedoutput: By implementing real world projects, you can enhance your data science skills.
paraphrasedoutput: Develop your data science skills by implementing real world projects.
paraphrasedoutput: By implementing real world projects, you can raise your data science skills.


In [4]:
# Constrained Beam search
force_words = ["improve"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=5,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Supercharge your data science skills by building real world projects.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills.
paraphrasedoutput: By executing real world projects, you can improve your data science skills.
paraphrasedoutput: Build real world projects to improve your data science skills.
paraphrasedoutput: Real world projects can help you improve your data science skills.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills.


In [5]:
# Constrained Beam search
force_words = ["upgrade"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Supercharge your data science skills by building real world projects.
paraphrasedoutput: By implementing real world projects, you can upgrade your data science skills.
paraphrasedoutput: By implementing real world projects, you can improve your data science skills. The upgrade to Data Science by the end of this year will be greatly appreciated!
paraphrasedoutput: By implementing real world projects, you can improve your data science skills. The upgrade to Data Science by the end of this year will be greatly enhanced!


In [6]:
# Copy writing example - Use a given word in paraphrasing

context = "Apple products show a great taste of design and user focus. One good example is the iPhone."

text = "paraphrase: "+context + " </s>"

input_ids = tokenizer(text, max_length=128, padding=True, return_tensors="pt").input_ids
input_ids = input_ids.to(device)

# Beam search

outputs = model.generate(
    input_ids,
    num_beams=10,
    num_return_sequences=3,
    max_length=128,
    early_stopping=True,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nNormal beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2280: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  "`max_length` is ignored when `padding`=`True` and there is no truncation strategy. "



Normal beam search

Original:  Apple products show a great taste of design and user focus. One good example is the iPhone.
paraphrasedoutput: Apple's iPhone is a good example.
paraphrasedoutput: Apple products are a good example of both form and customer attention.
paraphrasedoutput: Apple products are a good example of both design and user focus.


In [7]:
# Constrained Beam search
force_words = ["iPhone"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Apple products show a great taste of design and user focus. One good example is the iPhone.
paraphrasedoutput: The iPhone is a good example of design and user focus.
paraphrasedoutput: The iPhone is a good example of both design and user focus.
paraphrasedoutput: The iPhone is a good example.


In [8]:
# Constrained Beam search
force_words = ["iPhone", "customers"]
force_words_ids = tokenizer(force_words, add_special_tokens=False).input_ids

outputs = model.generate(
    input_ids,
    force_words_ids=force_words_ids,
    max_length=128,
    early_stopping=True,
    num_beams=10,
    num_return_sequences=3,
    no_repeat_ngram_size=1,
    remove_invalid_values=True,
)

print ("\nConstrained beam search\n")
print ("Original: ",context)
for beam_output in outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    print (sent)


Constrained beam search

Original:  Apple products show a great taste of design and user focus. One good example is the iPhone.
paraphrasedoutput: Apple's iPhone is a good example of both form and customers.
paraphrasedoutput: Apple's iPhone is a good example of both form and customer attention. The customers are always on alert when it comes to apple products, as shown by this one in the screenshot above (right).
paraphrasedoutput: Apple's iPhone is a good example of both form and customer attention. The customers are always on alert when it comes to apple products, as shown by this one in the image above (right).
